In [1]:
from pymut4se.api import discover

workspace = discover("../src")

print(workspace.project.name)
print(workspace.statistics())
print(workspace.statistics().as_dict())

src
ProjectStatistics(packages=7, modules=43, chunks=338, tests=138, tested_chunks=26, requirements=8)
{'packages': 7, 'modules': 43, 'original_chunks': 338, 'test_suites': 25, 'test_cases': 138, 'test_links': 317, 'chunks_with_tests': 26, 'requirements': 8}


In [2]:
packages = workspace.find_packages("pymut4se.mutation")
modules = workspace.find_modules("*generic*")
chunks = workspace.find_chunks("optional_parameter")


In [3]:
for operator in workspace.operators():
    print(operator.name, operator.class_name, operator.description)

add-if-not-null IfNotNullMutation Guard a function body so it only runs for non-null inputs.
arithmetic ArithmeticMutation Replace one arithmetic binary operator.
boolean-replacement BooleanReplacementMutation Invert one boolean literal.
constant-replacement ConstantReplacementMutation Transform one numeric constant.
control-replacement ControlReplacementMutation Swap break and continue.
delete-assignment DeleteAssignmentMutation Delete one assignment statement.
delete-decorator DeleteDecoratorMutation Delete one function decorator.
delete-if-statement DeleteIfStatementMutation Delete one complete if statement.
delete-return DeleteReturnMutation Delete one return statement.
delete-while DeleteWhileMutation Delete one complete while loop.
logical-connector LogicalConnectorMutation Swap and and or in one boolean expression.
optional-parameter-callee OptionalParamCalleeMutation Change an optional parameter default in the called function.
optional-parameter-caller OptionalParamCallerMutati

In [4]:
targets = [
    *workspace.find_modules("execution"),
    *workspace.find_chunks("_apply_mutation"),
]

new_mutants = workspace.mutate(
    targets,
    operators="all",
    max_degree=1,
)

Generating mutants: 1743 new | chunks processed: 61/61


In [9]:
mutants_without_tests = [
    mutant for mutant in new_mutants
    if not mutant.related_test_cases
]

len(mutants_without_tests)

1307

In [5]:
print(workspace.mutant_statistics())


MutantStatistics(total=1743, source_chunks=61, by_degree={1: 1743}, by_type={'arithmetic': 187, 'boolean_replacement': 29, 'cast_type': 456, 'comparison': 306, 'constant_replacement': 88, 'control_replacement': 2, 'delete_assignment': 128, 'delete_decorator': 14, 'delete_if_statement': 49, 'delete_return': 57, 'if_not_null': 152, 'logical_connector': 13, 'optional_param_callee': 20, 'return_pass': 39, 'swap_arguments': 149, 'unary': 54}, input_executions=0, test_executions=0, failed_tests=0)


In [6]:
arithmetic = workspace.find_mutants(
    "*",
    operator="Add",
    degree=1
)
print(len(arithmetic))

57


In [7]:
test_outputs = workspace.run_tests(
    new_mutants,
    parallel=True,
    max_workers=2,
    timeout_seconds=2,
)

Preparing execution environment...
Execution environment ready: C:\Users\matth\Documents\code\PyMut4SE\src\.pymut4se\venvs\edec7d7a7d86c52532e1b7f9cb918b276058f2fc3cc9dca0ede483e0c69113e8
Executing tests: 1743/1743


In [8]:
score = workspace.mutation_score()
print(score)

MutationScore(score=100.00%, killed=436, survived=0, untested=1307, incomplete=0, errors=0)
